# load data

In [1]:

import os
import pandas as pd
import numpy as np
import torch
import random
from sklearn import preprocessing
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import matthews_corrcoef, average_precision_score
from torch.optim import lr_scheduler
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist, squareform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import os
import pandas as pd

def load_data(base_path, file_names):
    """
    Load multiple CSV files from a specified base path and return a dictionary of DataFrames.

    Parameters:
    base_path (str): The base directory path where the CSV files are stored.
    file_names (list): A list of CSV file names to be loaded.

    Returns:
    dict: A dictionary where the keys are file names and the values are DataFrames.
    """
    data = {}
    
    for file_name in file_names:
        # 使用 os.path.join 来创建完整的文件路径
        full_path = os.path.join(base_path, file_name)
        
        # 使用 pandas 读取 csv 文件
        data[file_name] = pd.read_csv(full_path, index_col=0)
    
    return data

In [3]:

base_path = 'Code and data/Data/SysLM-I-input/BONUS-CF/data_CLR/'


file_names = ['otu_phylum_CLR.csv', 'otu_class_CLR.csv', 'otu_order_CLR.csv',
              'otu_family_CLR.csv', 'otu_genus_CLR.csv', 'otu_species_CLR.csv']


data = load_data(base_path, file_names)

otu_phylum = data['otu_phylum_CLR.csv']

otu_class = data['otu_class_CLR.csv']

otu_order = data['otu_order_CLR.csv']

otu_family = data['otu_family_CLR.csv']

otu_genus = data['otu_genus_CLR.csv']
otu_genus.shape 
otu_species = data['otu_species_CLR.csv']
otu_species.shape



print("otu_phylum shape:", otu_phylum.shape)
print("otu_class shape:", otu_class.shape)
print("otu_order shape:", otu_order.shape)
print("otu_family shape:", otu_family.shape)
print("otu_genus shape:", otu_genus.shape)
print("otu_species shape:", otu_species.shape)

otu_phylum shape: (8, 1744)
otu_class shape: (20, 1744)
otu_order shape: (36, 1744)
otu_family shape: (83, 1744)
otu_genus shape: (211, 1744)
otu_species shape: (602, 1744)


In [4]:

base_path1 = 'Code and data/Data/SysLM-I-input/BONUS-CF/all_time_RA/'

file_names1 = ['otu_phylum_na.csv', 'otu_class_na.csv', 'otu_order_na.csv', 'otu_family_na.csv', 'otu_genus_na.csv', 'otu_species_na.csv']

data = load_data(base_path1, file_names1)

otu_phylum_na = data['otu_phylum_na.csv']

otu_class_na = data['otu_class_na.csv']

otu_order_na = data['otu_order_na.csv']

otu_family_na = data['otu_family_na.csv']

otu_genus_na = data['otu_genus_na.csv']

otu_species_na = data['otu_species_na.csv']
otu_species_na.shape

(602, 1744)

# input data

In [5]:

import pandas as pd
import numpy as np
import torch

def Get_raw_mask_data(df1,df2,subject_number,time_length):

    column_groups1 = [list(range(i, i + time_length)) for i in range(0, df1.shape[1], time_length)]
    column_groups2 = [list(range(i, i + time_length)) for i in range(0, df2.shape[1], time_length)]


    small_dataframes1 = [df1.iloc[:, group] for group in column_groups1]
    small_dataframes2 = [df2.iloc[:, group] for group in column_groups2]

    big_df1 = pd.concat(small_dataframes1, axis=1)
    big_df2 = pd.concat(small_dataframes2, axis=1)
    numpy_array1 = big_df1.to_numpy()
    numpy_array2 = big_df2.to_numpy()

    small_arrays1 = np.split(numpy_array1, numpy_array1.shape[1] // time_length, axis=1)
    small_arrays2 = np.split(numpy_array2, numpy_array2.shape[1] // time_length, axis=1)


    rows1 = [row for small_array in small_arrays1 for row in small_array]
    rows2 = [row for small_array in small_arrays2 for row in small_array]


    rows_array1 = np.array(rows1)
    rows_array2 = np.array(rows2)
    
    regression_raw_data_ra1 = rows_array1.reshape((subject_number, df1.shape[0], time_length))
    regression_raw_data_clr2 = rows_array2.reshape((subject_number, df2.shape[0], time_length))

    mask = ~np.isnan(df1.to_numpy())

    mask = [pd.DataFrame(mask).iloc[:, group] for group in column_groups1]

    mask_array = np.array([df1.to_numpy() for df1 in mask])# 


    return regression_raw_data_ra1, regression_raw_data_clr2,mask_array

In [6]:
metric_type = 'P'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_phylum_na,otu_phylum,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

raw_data_ra.shape: (218, 8, 8)
raw_data_clr.shape: (218, 8, 8)
mask_data.shape: (218, 8, 8)


In [ ]:
metric_type = 'C'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_class_na,otu_class,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

In [ ]:
metric_type = 'O'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_order_na,otu_order,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

In [ ]:
metric_type = 'F'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_family_na,otu_family,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

In [ ]:
metric_type = 'G'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_genus_na,otu_genus,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

In [ ]:
metric_type = 'S'  #
datasetname = 'BONUS-CF'
raw_data_ra, raw_data_clr,mask_data = Get_raw_mask_data(otu_species_na,otu_species,subject_number=218,time_length=8)
print('raw_data_ra.shape:', raw_data_ra.shape)
print('raw_data_clr.shape:', raw_data_clr.shape)
print('mask_data.shape:', mask_data.shape)

# SysLM-I

In [7]:

import torch

def Get_input(data_raw,time_points,device):

    data_raw = torch.tensor(data_raw).to(device)

    batchsize = data_raw.shape[0]
    seq_num = data_raw.shape[1]
    seq_len = data_raw.shape[2]
    
   
    
    rows_array = data_raw.reshape((batchsize*seq_num, seq_len))

    time_array = np.array(time_points).reshape((1, 1, seq_len))  
    time_array = np.repeat(time_array, batchsize*seq_num, axis=0)  
    

    data = rows_array.reshape((batchsize, seq_num, 1, seq_len))

    time_array_reshaped = torch.tensor(time_array.reshape((batchsize, seq_num, 1, seq_len))).to(device)
    data_with_time = torch.cat((data, time_array_reshaped), axis=2)


    continuous_time_data_diff = torch.diff(data, axis=3)
    continuous_time_diff = torch.diff(time_array_reshaped, axis=3)

    data_with_data_time_gap = torch.cat((continuous_time_data_diff, continuous_time_diff), axis=2)


    subsequences = create_subsequences(rows_array)  

    data_subsequences = subsequences.reshape((batchsize, seq_num, 6, 3))#6,4,2,8
    
    return data_with_time, data_with_data_time_gap, data_subsequences

def create_subsequences(data, window_size=3, step_size=1):
    subsequences = []
    for sequence in data:

        for i in range(0, len(sequence) - window_size + 1, step_size):
            subsequences.append(sequence[i:i+window_size])
    return torch.stack(subsequences)

In [8]:
data, data_with_time_diff, data_subseq = Get_input(raw_data_clr,time_points = [3,4,5,6,8,10,12,15],device=device)#[4, 7, 10, 13, 16, 19, 22,28]
print('data.shape:', data.shape)
print('data_with_time_diff.shape:', data_with_time_diff.shape)
print('data_subseq.shape:', data_subseq.shape)

data.shape: torch.Size([218, 8, 2, 8])
data_with_time_diff.shape: torch.Size([218, 8, 2, 7])
data_subseq.shape: torch.Size([218, 8, 6, 3])


In [9]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader,Dataset

class MyDataset1(Dataset):
    def __init__(self, data1, data2, mask_array, one_hot_encoded_data):
        self.data1 = torch.tensor(data1, dtype=torch.float32)
        self.data2 = torch.tensor(data2, dtype=torch.float32)
        self.mask_array = torch.tensor(mask_array, dtype=torch.bool)  # 
        self.one_hot_encoded_data = torch.tensor(one_hot_encoded_data, dtype=torch.float32)
        
    def __len__(self):
        return len(self.data1)
    
    def __getitem__(self, index):
        return self.data1[index], self.data2[index], self.mask_array[index],self.one_hot_encoded_data[index]

def collate_fn1(batch):
    data1, data2, mask_array,one_hot_encoded_data = zip(*batch)
    data1 = torch.stack(data1)
    data2 = torch.stack(data2)
    mask_array = torch.stack(mask_array)  
    one_hot_encoded_data = torch.stack(one_hot_encoded_data)
    
    return data1, data2, mask_array, one_hot_encoded_data

In [10]:

sg = pd.read_csv("Code and data/Data/metadata/BONUS-CF/SG.csv",index_col=0)
# sc = pd.read_csv("Code and data/Data/metadata/BONUS-CF/SC.csv",index_col=0)
factors1 = np.array(sg)
factors = torch.from_numpy(factors1)
factors.shape

torch.Size([218, 3])

In [11]:

def shannon_index(samples):

    p = samples + 1e-10  # avoid log(0)
    shannon_entropy = -torch.sum(p * torch.log(p), dim=1)
    shannon_entropy = shannon_entropy.unsqueeze(1)#
    return shannon_entropy


def bray_curtis(samples):
    samples_np = samples.detach().cpu().numpy()  # 
    distances_np = pairwise_distances(samples_np, metric='braycurtis')  #  Bray-Curtis 
    distances = torch.from_numpy(distances_np).to(samples.device)  # 
    return distances


def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

def Frobenius_loss(A, B):

    loss = torch.norm(A - B, p='fro')

    return loss


# 训练
def Infer_train(model, Epoch_num, Batch_size, learning_rate, train_loader, test_loader,lam1,lam2, device):

    train_losses = []
    train_mses = []
    train_rmses = []
    train_maes = []
    train_r2s = []
    epochs = []
    losses = []
    lr_list = [] 
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)#, weight_decay=1e-4
    scheduler = lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.1)
#     scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, 'min')
    
    model.train()
    for epoch in range(Epoch_num):

        train_loss = 0.0
        train_mse = 0.0
        train_rmse = 0.0
        train_mae = 0.0    
        train_r2 = 0.0
        
#         confounders
        for step, (batch_x,batch_x_ra,batch_mask,batch_factor) in enumerate(train_loader):
            optimizer.zero_grad() 
            
            train_input = batch_x.to(device)
            train_ra = batch_x_ra.to(device)
            train_mask = batch_mask.to(device)
            train_confounders = batch_factor.to(device)
            y_pred, y_replaced = model(train_input,train_ra, train_mask, train_confounders)#

            y_pred_masked = torch.masked_select(y_pred, train_mask)
            train_input_masked = torch.masked_select(train_ra, train_mask)

            loss1 = F.mse_loss(y_pred_masked, train_input_masked)
      
            otu_num = train_mask.shape[1]
#             print(otu_num)
            train_mask1 = train_mask.permute(0,2,1).reshape(-1, otu_num)  # 
            y_pred_masked1 = y_pred.permute(0,2,1).reshape(-1, otu_num)  # 
            train_input_masked1 = train_ra.permute(0,2,1).reshape(-1, otu_num)  

            first_col_mask = train_mask1[:, 0]

            train_input_d = train_input_masked1[first_col_mask]
            y_pred_d = y_pred_masked1[first_col_mask]
            
            alphad1 = shannon_index(train_input_d)
            alphad2 = shannon_index(y_pred_d)
            betad1 = bray_curtis(train_input_d)

            betad2 = bray_curtis(y_pred_d)
            loss2 = Frobenius_loss(alphad1,alphad2)
            loss3 = Frobenius_loss(betad1, betad2)    

#             lambda1 = loss1 / (loss1+loss2+loss3)
            lambda1 = lam1 #loss2 / (loss1+loss2+loss3)
            lambda2 = lam2 #loss3 / (loss1+loss2+loss3)
            loss = loss1+ lambda1*loss2+lambda2*loss3
#             loss = loss1+ loss2+loss3
            tra_pred = y_pred_masked.detach().cpu().numpy()
            tra_true = train_input_masked.detach().cpu().numpy()
            tra_mse = mean_squared_error(tra_true, tra_pred)
            tra_rmse = rmse(tra_true, tra_pred)
            tra_mae = mean_absolute_error(tra_true, tra_pred)
            tra_r2 = r2_score(tra_true, tra_pred)

            loss.backward()
            
            optimizer.step()          
            train_loss += loss.item()
            train_mse += tra_mse
            train_rmse += tra_rmse
            train_mae += tra_mae
            train_r2 += tra_r2
            
        train_loss /= len(train_loader)
        scheduler.step()
        
        train_mse /= len(train_loader)
        train_rmse /= len(train_loader)
        train_mae /= len(train_loader)
        train_r2 /= len(train_loader)
        
#         val_loss, val_mse, val_rmse, val_mae,val_r2 = Infer_test(model, Batch_size, test_loader,lam1,lam2,device)

        train_losses.append(train_loss)
        train_mses.append(train_mse)
        train_rmses.append(train_rmse)
        train_maes.append(train_mae)
        train_r2s.append(train_r2)
        epochs.append(epoch)
        lr_list.append(optimizer.param_groups[0]['lr'])  
        
        if epoch % 50 == 0: 
            print(f'Epoch: {epoch}, '
                  f'train_Loss: {train_loss:.5f}, '
                  f'train_MSE: {train_mse:.5f}, '
                  f'train_RMSE: {train_rmse:.5f}, '
                  f'train_MAE: {train_mae:.5f}, '
                  f'train_R2: {train_r2:.5f}')
            
#             print(f'Epoch: {epoch}, '
#                   f'val_Loss: {val_loss:.5f}, '
#                   f'val_MSE: {val_mse:.5f}, '
#                   f'val_RMSE: {val_rmse:.5f}, '
#                   f'val_MAE: {val_mae:.5f}, '
#                   f'val_R2: {val_r2:.5f}')

    if test_loader is not None:
        eval_val_loss, eval_val_mse, eval_val_rmse, eval_val_mae,eval_val_r2 = Infer_test(model, Batch_size, test_loader,lam1,lam2,device)
    else:
        eval_val_loss, eval_val_mse, eval_val_rmse, eval_val_mae, eval_val_r2 = None, None, None, None, None

    return eval_val_loss, eval_val_mse, eval_val_rmse, eval_val_mae,eval_val_r2


def Infer_test(model, Batch_size, test_loader,lam1,lam2,device):  #
    model.eval()
    with torch.no_grad():  # 
        test_loss = 0.0
        test_mse = 0.0
        test_rmse = 0.0
        test_mae = 0.0
        test_r2 = 0.0               
        for index, (batch_x, batch_x_ra, batch_mask, batch_factor) in enumerate(test_loader):
            test_input = batch_x.to(device)
            test_ra = batch_x_ra.to(device)
            test_mask = batch_mask.to(device)
            test_confounders = batch_factor.to(device)
            test_pred,test_relaced = model(test_input,test_ra, test_mask, test_confounders)

            test_outputs_masked = torch.masked_select(test_pred, test_mask)
            test_input_masked = torch.masked_select(test_ra, test_mask)


            loss1 = F.mse_loss(test_outputs_masked, test_input_masked) 

      
            otu_num = test_mask.shape[1]
            test_mask1 = test_mask.permute(0,2,1).reshape(-1, otu_num)  
            y_pred_masked1 = test_pred.permute(0,2,1).reshape(-1, otu_num) 
            test_input_masked1 = test_ra.permute(0,2,1).reshape(-1, otu_num)  

            first_col_mask = test_mask1[:, 0]
            # 
            input_d = test_input_masked1[first_col_mask]
            y_pred_d = y_pred_masked1[first_col_mask]

            alphad1 = shannon_index(input_d)
            alphad2 = shannon_index(y_pred_d)
            betad1 = bray_curtis(input_d)
            betad2 = bray_curtis(y_pred_d)  
            loss2 = Frobenius_loss(alphad1,alphad2)
            loss3 = Frobenius_loss(betad1, betad2)  
#             lambda1 = loss1 / (loss1+loss2+loss3)
            lambda2 = lam1 #loss2 / (loss1+loss2+loss3)
            lambda3 = lam2 #loss3 / (loss1+loss2+loss3)
            val_loss = loss1+ lambda2*loss2+lambda3*loss3

            test_pred = test_outputs_masked.detach().cpu().numpy()
            test_true = test_input_masked.detach().cpu().numpy()
            val_mse = mean_squared_error(test_true, test_pred)
            val_rmse = rmse(test_true, test_pred)
            val_mae = mean_absolute_error(test_true, test_pred)
            val_r2 = r2_score(test_true, test_pred)


            test_loss += val_loss.item()
            test_mse += val_mse
            test_rmse += val_rmse
            test_mae += val_mae
            test_r2 += val_r2
   

        test_loss /= len(test_loader)
        test_mse /= len(test_loader)
        test_rmse /= len(test_loader)
        test_mae /= len(test_loader)
        test_r2 /= len(test_loader)
           
    return test_loss, test_mse, test_rmse, test_mae, test_r2

In [12]:

from torch.nn.utils import weight_norm

torch.manual_seed(32)
torch.cuda.manual_seed_all(32)
np.random.seed(32)
random.seed(32)
# torch.backends.cudnn.deterministic = True

 
def inverse_clr_transform(y):
    exp_y = torch.exp(y)
    return exp_y / torch.sum(exp_y, dim = 1, keepdim = True)  

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)

        self.downsample = nn.Conv1d(
            n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, dropout, kernel_size=2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = self.network(x)
        return x

class TCN(nn.Module):
    def __init__(self, input_channels, output_channels, dropout):
        super(TCN, self).__init__()
   
        self.tcn = nn.Sequential(TemporalConvNet(input_channels, output_channels, dropout=dropout),
                                  nn.ReLU(),
                                  nn.AdaptiveAvgPool1d(6))#6,4,2,8
    def forward(self, x):
        out = self.tcn(x)
        return out
    
class TCN_Combination(nn.Module):
    def __init__(self, input_channels1, input_channels2, input_channels3, output_channels, dropout):
        super(TCN_Combination, self).__init__()
        self.tcn1 = TCN(input_channels1, output_channels, dropout=dropout)
        self.tcn2 = TCN(input_channels2, output_channels, dropout=dropout)
        self.tcn3 = TCN(input_channels3, output_channels, dropout=dropout)
        
    def forward(self, *inputs):
        x1, x2, x3 = inputs
        x1 = x1.reshape(-1, *x1.shape[2:])## 
        x2 = x2.reshape(-1, *x2.shape[2:])#
        x3 = x3.reshape(-1, *x3.shape[2:])
        x3 = x3.permute(0, 2, 1)## 
        tcn_out1 = self.tcn1(x1)
        tcn_out2 = self.tcn2(x2)
        tcn_out3 = self.tcn3(x3)        
        tcn_out = torch.cat((tcn_out1, tcn_out2, tcn_out3), dim=1)  
 
        return tcn_out
     
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(BiLSTM, self).__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size * 2) 
    
    def forward(self, x):
        out, _ = self.bilstm(x)
        out = self.layer_norm(out)
        return out, _

class WeightedFusion(nn.Module):
    def __init__(self, input_dim, num_sources):
        super(WeightedFusion, self).__init__()
        self.weights = nn.Parameter(torch.ones(num_sources, requires_grad=True))

    def forward(self, *inputs):
        assert len(inputs) == len(self.weights)
        weighted_sum = sum(w * x for w, x in zip(self.weights, inputs))

        return weighted_sum / self.weights.sum()   
    
class BiLSTM_Attention_Weighted(nn.Module):
    def __init__(self, input_size1, input_size2, input_size3,input_size4, hidden_size):
        super(BiLSTM_Attention_Weighted, self).__init__()
        self.bilstm1 = BiLSTM(input_size1, hidden_size)
        self.bilstm2 = BiLSTM(input_size2, hidden_size)
        self.bilstm3 = BiLSTM(input_size3, hidden_size)
        self.bilstm4 = BiLSTM(input_size4, hidden_size)
        self.weighted_fusion = WeightedFusion(input_dim=hidden_size*2, num_sources=4)
        
    def attention_net(self, lstm_output, final_state):
        batch_size = len(lstm_output)
        hidden = final_state.view(batch_size, -1, 1) 
        attn_weights = torch.bmm(lstm_output, hidden).squeeze(2)  
        soft_attn_weights = F.softmax(attn_weights, 1)
   
        context = torch.bmm(lstm_output.transpose(
            1, 2), soft_attn_weights.unsqueeze(2)).squeeze(2)
        return context, soft_attn_weights  
        
    def forward(self, *inputs):
        tcn_x, x1, x2, x3 = inputs

        tcn_x = tcn_x.reshape(-1,1,32*3*6)  
        x1 = x1.reshape(-1, *x1.shape[2:])## Output shape: 
        x2 = x2.reshape(-1, *x2.shape[2:])## Output shape: 
        x3 = x3.reshape(-1, *x3.shape[2:])## Output shape:   
        x1 = x1.permute(0, 2, 1)## Output shape:
        x2 = x2.permute(0, 2, 1)##      
        lstm_tcn_out, (final_hidden_state, final_cell_state) = self.bilstm1(tcn_x) 
        lstm_out_x1, (final_hidden_state1, final_cell_state1) = self.bilstm2(x1) 
        lstm_out_x2, (final_hidden_state2, final_cell_state2) = self.bilstm3(x2) 
        lstm_out_x3, (final_hidden_state3, final_cell_state3) = self.bilstm4(x3) 

        attn_tcn, attention = self.attention_net(lstm_tcn_out, final_hidden_state)
        attn_lstm1, attention1 = self.attention_net(lstm_out_x1, final_hidden_state1)
        attn_lstm2, attention2 = self.attention_net(lstm_out_x2, final_hidden_state2)
        attn_lstm3, attention3 = self.attention_net(lstm_out_x3, final_hidden_state3)        

        weighted_attn_out = self.weighted_fusion(attn_tcn, attn_lstm1, attn_lstm2, attn_lstm3)
    
        return weighted_attn_out
class ImputationModel(nn.Module):
    def __init__(self,tcn_params, bilstm_params,fc_params1, fc_params2, dropout,time_points, device):
        super(ImputationModel, self).__init__()
        self.TCN_model = TCN_Combination(**tcn_params,dropout=dropout)
        self.BiLSTM_model = BiLSTM_Attention_Weighted(**bilstm_params)
        self.confounder_fc1 = nn.Sequential(
            nn.Linear(*fc_params1["linear_dims"]),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.fc_regression = nn.Sequential(
            nn.Linear(*fc_params2["linear_dims"])
#             nn.Sigmoid()
        )
        self.time_points = time_points
        self.device = device

    def forward(self, *inputs):
        x, x_ra, mask,confounders = inputs

        batch_size, seq_num, seq_len = x.size()
        x1,x2,x3 = Get_input(x,self.time_points,self.device)        
      
        tcn_out = self.TCN_model(x1,x2,x3)     
        lstm_out = self.BiLSTM_model(tcn_out,x1,x2,x3)
        lstm_out = lstm_out.view(batch_size, seq_num, -1)#
        conf_embed = self.confounder_fc1(confounders)
        conf_embed = conf_embed.unsqueeze(1).expand(-1, seq_num, -1)#
        reg_input = torch.cat((lstm_out,conf_embed),dim=2)#
        regression_out = self.fc_regression(reg_input) 
        regression_out = inverse_clr_transform(regression_out)

        regression_masked = torch.where(mask, x_ra, regression_out) #替换RA      


        return regression_out, regression_masked


# cross validation

In [13]:
import torch
import collections
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings("ignore")


torch.manual_seed(32)
torch.cuda.manual_seed_all(32)
np.random.seed(32)
random.seed(32)
# torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.deterministic = True

# Define hyperparameters
Epoch_num = 200
Batch_size = 32
learning_rate = 0.001  # [0.01,0.001,0.0001,0.00001]
time_points = [3,4,5,6,8,10,12,15]

tcn_params = {
    "input_channels1": 2,
    "input_channels2": 2,
    "input_channels3": 3,
    "output_channels": [32, 32, 32]
}

bilstm_params = {
    "input_size1": 32*3*6,
    "input_size2": 2,
    "input_size3": 2,
    "input_size4": 3,
    "hidden_size": 64
}


fc_params1 = {
    "linear_dims": (factors.shape[1], 128)
}

fc_params2 = {
    "linear_dims": (256, len(time_points))#8,6,4,10
}
dropout=0.5



history = collections.defaultdict(list)


kf = KFold(n_splits=5, shuffle=True, random_state=36)


for i, (train_index, test_index) in enumerate(kf.split(raw_data_clr)):
    print('*'*30, '第', i + 1, '折', '*'*30)
 
    train_data, train_data_ra, train_mask, train_factor = raw_data_clr[ train_index], raw_data_ra[ train_index], mask_data[train_index], factors[train_index]

    test_data, test_data_ra, test_mask, test_factor = raw_data_clr[test_index], raw_data_ra[test_index], mask_data[test_index], factors[test_index]

    train_torch_dataset = MyDataset1(train_data, train_data_ra,train_mask, train_factor)
    train_loader = DataLoader(dataset=train_torch_dataset, batch_size=Batch_size,
                              shuffle=True, num_workers=0, collate_fn=collate_fn1) 

    test_torch_dataset = MyDataset1(test_data,test_data_ra, test_mask, test_factor)
    test_loader = DataLoader(dataset=test_torch_dataset, batch_size=Batch_size,
                             shuffle=True, num_workers=0, collate_fn=collate_fn1)  

    
    Net = ImputationModel(tcn_params, bilstm_params, fc_params1, fc_params2, dropout, time_points, device).to(device)

  
    test_loss, test_mse, test_rmse, test_mae, test_r2 = Infer_train(Net, Epoch_num, Batch_size, learning_rate, train_loader, test_loader,lam1=1e-5,lam2=1e-5,device=device)

    print(f'test_loss: {test_loss}, '
          f'test_mse: {test_mse}, '
          f'test_rmse: {test_rmse}, '
          f'test_mae: {test_mae}, '
          f'test_r2: {test_r2}')
    
    history['test_loss'].append(test_loss)
    history['test_mse'].append(test_mse)
    history['test_rmse'].append(test_rmse)
    history['test_mae'].append(test_mae)
    history['test_r2'].append(test_r2)

print('Average Loss:', np.mean(history['test_loss']),
      'Average MSE:', np.mean(history['test_mse']),
      'Average RMSE:', np.mean(history['test_rmse']),
      'Average MAE:', np.mean(history['test_mae']),
      'Average R2:', np.mean(history['test_r2']))

****************************** 第 1 折 ******************************
Epoch: 0, train_Loss: 0.05476, train_MSE: 0.05376, train_RMSE: 0.23182, train_MAE: 0.17087, train_R2: 0.00989
Epoch: 50, train_Loss: 0.00111, train_MSE: 0.00098, train_RMSE: 0.03128, train_MAE: 0.01864, train_R2: 0.98171
Epoch: 100, train_Loss: 0.00043, train_MSE: 0.00035, train_RMSE: 0.01874, train_MAE: 0.01196, train_R2: 0.99354
Epoch: 150, train_Loss: 0.00038, train_MSE: 0.00030, train_RMSE: 0.01738, train_MAE: 0.01121, train_R2: 0.99440
test_loss: 0.0004032090550037017, test_mse: 0.000339424135745503, test_rmse: 0.018415779806673527, test_mae: 0.011604099068790674, test_r2: 0.993951731591781
****************************** 第 2 折 ******************************
Epoch: 0, train_Loss: 0.05478, train_MSE: 0.05381, train_RMSE: 0.23190, train_MAE: 0.17060, train_R2: 0.01387
Epoch: 50, train_Loss: 0.00123, train_MSE: 0.00110, train_RMSE: 0.03313, train_MAE: 0.01959, train_R2: 0.97962
Epoch: 100, train_Loss: 0.00050, train_M